## Project Planning Stage (Individual)
#### Peter Wojnicki | 78625613 | Group 9 | Section 008

### <u>(1) Data Description:</u>

<center><b>Table 1.</b> Overview of Data with Number of Columns and Observations</center>

| Source File Name | Number of Columns | Total Number of Observations |
|:-:|:-:|:-:|
|players.csv| 7 | 196 |
|sessions.csv| 5 | 1535 | 

#### **Players Dataset**

**experience (chr):** "Amateur", "Beginner", "Regular", "Pro", "Veteran" assigned based on experience.

**subscribe (lgl):** "True" or "False" if subscribed.

**hashedEmail (chr):** Hashed Email of player.

**played_hours (dbl):** Number of hours played.

**name (chr):** Name of player.

**gender (chr):** "Female" or "Male" for gender of player.

**Age (dbl):** Age of player.

#### Potential Issues:
- experience is self-declared and might have some bias
- joining players with sessions 

#### **Sessions Dataset**

**hashedEmail (chr):** Hashed Email of player.

**start_time (chr):** Start time of session in format of DAY/MONTH/YEAR HOUR:MINUTE.

**end_time (chr):** End time of session in format of DAY/MONTH/YEAR HOUR:MINUTE.

**original_start_time (dbl):** Start time recorded in UNIX time (milliseconds).

**original_end_time (dbl):** End time recorded in UNIX time (milliseconds).

#### Potential Issues:
- start_time and end_time are dates and times that both need to be wrangled into a usable form
- original_start_time and original_end_time are in milliseconds

### <u>(2) Questions:</u>

I hope to address demand forecasting and try to use this data to predict highest demand periods.

### <u>(3) Exploratory Data Analysis and Visualization</u>

<center><b>Table 2.</b> Summary statistics for variables of interest</center>

| Variable Name | Mean | Median | Minimum | Maximum | Number of Observations |
|:-:|:-:|:-:|:-:|:-:|:-:|
|Player Age (years) | 21.14 | 19 | 9 | 58 | 196 |
|Played Hours (hours) | 5.85 | 0.1 | 0 | 223.1 | 196 |
|Elapsed Session Time (minutes) | 50.86 | 30 | 3 | 259 | 1535 |


No duplicated names or hashed emails in players data (checked with duplicated function) to avoid any clashing when joining data.


In [ ]:
### Run this cell before continuing.
library(tidyverse)

#### Read the Datasets from URLs

In [ ]:
players_data <- read_csv("https://raw.githubusercontent.com/wojpc/wojpc-dsci100-project-008-09/85fbc982690237a92b10d714a1b540644c562325/data/players.csv")
sessions_data <- read_csv("https://raw.githubusercontent.com/wojpc/wojpc-dsci100-project-008-09/85fbc982690237a92b10d714a1b540644c562325/data/sessions.csv")

#### Number of Columns and Preview Data

In [ ]:
nrow(players_data)
nrow(sessions_data)

head(players_data)
head(sessions_data)

#### Check for Duplicated Names or Emails in Players Data

In [ ]:
dups_name <- duplicated(players_data$name)
dups_hashed <- duplicated(players_data$hashedEmail)

#### Summary Statistics for Quantitative Variables in Players Dataset

In [ ]:
players_played_mean <- mean(players_data$played_hours, na.rm = TRUE)
players_played_min <- min(players_data$played_hours, na.rm = TRUE)
players_played_max <- max(players_data$played_hours, na.rm = TRUE)
players_played_med <- median(players_data$played_hours, na.rm = TRUE)

players_age_mean <- mean(players_data$Age, na.rm = TRUE)
players_age_min <- min(players_data$Age, na.rm = TRUE)
players_age_max <- max(players_data$Age, na.rm = TRUE)
players_age_med <- median(players_data$Age, na.rm = TRUE)

players_played_mean
players_played_med
players_played_min
players_played_max

players_age_mean
players_age_med
players_age_min
players_age_max

#### Grouping Qualitative Variables

In [ ]:
# Aggregate by experience level
players_exp <- players_data |>
    group_by(experience) |>
    summarize(total_exp = n())

# Aggregate by subscription status
players_subbed <- players_data |>
    group_by(subscribe) |>
    summarize(total_exp = n())

# Aggregate by gender
players_gender <- players_data |>
    group_by(gender) |>
    summarize(total_exp = n())

# Check players who played more than once
sessions_per_player <- sessions_players_elapsed |>
    group_by(name) |>
    summarize(total_exp = n())

#### Left Join Data on Hashed Email and Wrangle Time
A new column called session_time is added to show elapsed time of session and times are converted from strings to more usable data. Sessions data is now combined with player information. original_start_time and original_end_time were useless so I removed them to make the table tidier and less redundant. 

In [ ]:
sessions_players_joined <- sessions_data |>
  left_join(players_data, by = "hashedEmail")

sessions_players_elapsed <- sessions_players_joined |>
    mutate(end_time = as.POSIXct(end_time, format = "%d/%m/%Y %H:%M"),
           start_time =  as.POSIXct(start_time, format = "%d/%m/%Y %H:%M")) |>
    mutate(session_time_elapsed = as.numeric(end_time - start_time)) |>
    select(-hashedEmail, -original_start_time, -original_end_time)

nrow(sessions_players_elapsed)
head(sessions_players_elapsed)

#### Summary Statistics for Elapsed Session Time in Joined Dataset

In [ ]:
sessions_players_elapsed_mean <- mean(sessions_players_elapsed$session_time_elapsed, na.rm = TRUE)
sessions_players_elapsed_min <- min(sessions_players_elapsed$session_time_elapsed, na.rm = TRUE)
sessions_players_elapsed_max <- max(sessions_players_elapsed$session_time_elapsed, na.rm = TRUE)
sessions_players_elapsed_med <- median(sessions_players_elapsed$session_time_elapsed, na.rm = TRUE)

sessions_players_elapsed_mean
sessions_players_elapsed_med 
sessions_players_elapsed_min 
sessions_players_elapsed_max

#### Histograms for Quantitative Date

In [ ]:
age_plot <- hist(players_data$Age)

played <- hist(players_data$played_hours)

sess <- hist(sessions_players_elapsed$session_time_elapsed)

age
played
sess